# 🚀 Official CVPR 2024 SED Pipeline: Baseline vs. SED + CM-GF
### *Full OpenCLIP Text-Visual Evaluation on Official ADE20K Benchmark*

This notebook runs the **full official SED pipeline** using Detectron2, OpenCLIP text embeddings, and the authors' official pre-trained checkpoint (`sed_convnext_b.pth`).

**Key Highlights**:
1. Evaluates on 100% of the official ADE20K validation benchmark.
2. Loads official pre-trained ConvNeXt-B weights directly.
3. Uses the authentic CLIP text prompt ensemble across all 150 classes.
4. Compares **Baseline SED (31.8% mIoU)** directly against **SED + CM-GF (Ours)**.

👉 Run the cells sequentially from top to bottom.

In [ ]:
# Cell 1: Check GPU & Install Detectron2 and Core Dependencies
!nvidia-smi
!pip install -q ninja cython timm==1.0.30 einops ftfy gdown
# Install detectron2 with --no-build-isolation so it can link against Colab's pre-installed PyTorch
!pip install --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import detectron2
print('✅ Detectron2 version:', detectron2.__version__)
print('✅ Environment ready.')

In [ ]:
# Cell 2: Clone Official SED Repository & Download Pre-Trained Checkpoint
import os
if not os.path.exists('SED'):
    !git clone https://github.com/xb534/SED.git
%cd SED

# Install author's modified OpenCLIP (which enables dense=True feature map extraction)
!pip install -q -e open_clip/

# Download official CVPR 2024 ConvNeXt-B checkpoint (from author's Google Drive)
if not os.path.exists('sed_convnext_b.pth'):
    print('Downloading official SED ConvNeXt-B checkpoint...')
    !gdown --id 1qx6zGZgSPkF6TObregRz4uzQqSRHrgUw -O sed_convnext_b.pth
print('✅ Codebase, dense OpenCLIP, and checkpoint ready.')

In [ ]:
# Cell 3: Download & Prepare Official ADE20K-150 Dataset
!mkdir -p datasets
if not os.path.exists('datasets/ADEChallengeData2016'):
    print('Downloading official MIT CSAIL ADE20K dataset (takes ~12s on Colab)...')
    !cd datasets && wget -q http://data.csail.mit.edu/places/ADEchallenge/ADEChallengeData2016.zip
    !cd datasets && unzip -q -o ADEChallengeData2016.zip
    print('Preparing Detectron2 label format for ADE20K...')
    !python datasets/prepare_ade20k_150.py
print('✅ ADE20K dataset ready in datasets/ADEChallengeData2016!')

In [ ]:
# Cell 4: Inject Tested Cross-Modal Guided Filter (CM-GF) into Cloned SED Codebase
# 1. Write cm_gf.py
with open('sed/modeling/transformer/cm_gf.py', 'w', encoding='utf-8') as f:
    f.write("\"\"\"\nCross-Modal Guided Filter (CM-GF) / Adaptive Guided Fusion Module (AGFM)\n\nImplemented for Open-Vocabulary Semantic Segmentation refinement (SED CVPR 2024 extension).\nUses shallow visual features as guidance to sharpen blurry semantic cost maps via closed-form,\ndifferentiable local linear transformations with O(N) box filtering.\n\nPhase 2 Standalone Deliverables:\n1. FastBoxFilter: Efficient nn.Module box filtering layer using F.avg_pool2d\n2. GuidedFilterCore: Deterministic parameter-free closed-form guided filter\n3. CMGFModule: Adaptive multi-mode guided filter with residual gating\n4. AGFM: Composite Adaptive Guided Fusion Module for end-to-end skip connection fusion\n\"\"\"\n\nfrom typing import Optional\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef fast_box_filter(x: torch.Tensor, radius: int) -> torch.Tensor:\n    \"\"\"\n    Computes 2D box filter (local window mean) with O(N) complexity using F.avg_pool2d.\n    \n    Args:\n        x: Input tensor of shape (B, C, H, W)\n        radius: Radius of the filter window. Window size will be 2 * radius + 1.\n        \n    Returns:\n        Filtered tensor of shape (B, C, H, W) with preserved spatial dimensions.\n    \"\"\"\n    kernel_size = 2 * radius + 1\n    # count_include_pad=False ensures correct boundary averaging without dark border artifacts\n    return F.avg_pool2d(\n        x,\n        kernel_size=kernel_size,\n        stride=1,\n        padding=radius,\n        count_include_pad=False\n    )\n\n\nclass FastBoxFilter(nn.Module):\n    \"\"\"\n    Box filter nn.Module wrapper around fast_box_filter.\n    Enables parameterized placement in sequential networks.\n    \"\"\"\n\n    def __init__(self, radius: int = 2):\n        super().__init__()\n        self.radius = radius\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return fast_box_filter(x, self.radius)\n\n    def extra_repr(self) -> str:\n        return f\"radius={self.radius}, window_size={2 * self.radius + 1}\"\n\n\nclass GuidedFilterCore(nn.Module):\n    \"\"\"\n    Deterministic, parameter-free closed-form Guided Filter.\n    Computes:\n        q_i = a_bar_i * I_i + b_bar_i\n    where (a_k, b_k) are analytical regression coefficients minimizing:\n        E(a_k, b_k) = sum_{i in w_k} ((a_k * I_i + b_k - P_i)^2 + eps * a_k^2)\n    \"\"\"\n\n    def __init__(self, radius: int = 2, eps: float = 1e-3):\n        super().__init__()\n        self.radius = radius\n        self.eps = eps\n\n    def forward(self, guidance: torch.Tensor, cost_map: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Args:\n            guidance (I): Guidance feature map (B, C, H, W) or (B, 1, H, W)\n            cost_map (P): Input cost / semantic probability map to be refined (B, C, H, W)\n            \n        Returns:\n            Refined cost map (q) of shape (B, C, H, W) with aligned physical edges.\n        \"\"\"\n        # 1. Compute local window statistics via O(N) box filtering\n        mean_I = fast_box_filter(guidance, self.radius)\n        mean_P = fast_box_filter(cost_map, self.radius)\n\n        corr_I = fast_box_filter(guidance * guidance, self.radius)\n        corr_IP = fast_box_filter(guidance * cost_map, self.radius)\n\n        # 2. Local variance and cross-covariance with non-negative clamping\n        var_I = corr_I - mean_I * mean_I\n        var_I = torch.clamp(var_I, min=0.0)  # Numerical safeguard against tiny negative FP precision errors\n        cov_IP = corr_IP - mean_I * mean_P\n\n        # 3. Closed-form linear coefficients (a_k, b_k)\n        # eps prevents division by zero in homogeneous/flat regions\n        a = cov_IP / (var_I + self.eps)\n        b = mean_P - a * mean_I\n\n        # 4. Overlapping window averaging\n        mean_a = fast_box_filter(a, self.radius)\n        mean_b = fast_box_filter(b, self.radius)\n\n        # 5. Linear reconstruction: q = a_bar * I + b_bar\n        q = mean_a * guidance + mean_b\n        return q\n\n    def extra_repr(self) -> str:\n        return f\"radius={self.radius}, eps={self.eps}\"\n\n\nclass CMGFModule(nn.Module):\n    \"\"\"\n    Cross-Modal Guided Filter Module (CMGFModule) with channel projection and residual gating.\n    \n    Can handle different channel dimensions between guidance (shallow visual features)\n    and cost map (upsampled semantic features), and incorporates a learnable residual gate\n    so the network can gracefully interpolate between original features and filtered edges.\n    \"\"\"\n\n    def __init__(\n        self,\n        guidance_channels: int,\n        cost_map_channels: int,\n        radius: int = 2,\n        eps: float = 1e-3,\n        mode: str = \"channelwise\",\n        use_residual_gate: bool = True\n    ):\n        \"\"\"\n        Args:\n            guidance_channels: Number of channels in guidance feature map I.\n            cost_map_channels: Number of channels in cost map P.\n            radius: Radius of the filtering window (kernel size = 2 * radius + 1).\n            eps: Regularization parameter penalizing large linear coefficients.\n            mode: 'channelwise' (full projection to cost_map_channels) or \n                  'spatial' (ultra-lightweight 1-channel edge guidance map, ideal for low GPU power).\n            use_residual_gate: If True, uses learnable gamma to smoothly blend original and refined maps.\n        \"\"\"\n        super().__init__()\n        self.radius = radius\n        self.eps = eps\n        self.mode = mode\n        self.use_residual_gate = use_residual_gate\n\n        if mode == \"channelwise\":\n            if guidance_channels != cost_map_channels:\n                self.guidance_proj = nn.Sequential(\n                    nn.Conv2d(guidance_channels, cost_map_channels, kernel_size=1, bias=False),\n                    nn.BatchNorm2d(cost_map_channels),\n                    nn.ReLU(inplace=True)\n                )\n            else:\n                self.guidance_proj = nn.Identity()\n        elif mode == \"spatial\":\n            # Ultra-lightweight: compresses guidance into 1 spatial edge channel\n            self.guidance_proj = nn.Sequential(\n                nn.Conv2d(guidance_channels, 8, kernel_size=1, bias=False),\n                nn.BatchNorm2d(8),\n                nn.ReLU(inplace=True),\n                nn.Conv2d(8, 1, kernel_size=3, padding=1, bias=False),\n                nn.Sigmoid()\n            )\n        else:\n            raise ValueError(f\"Unknown mode: {mode}. Choose 'channelwise' or 'spatial'.\")\n\n        self.filter_core = GuidedFilterCore(radius=radius, eps=eps)\n\n        if self.use_residual_gate:\n            # Learnable scalar gating parameter initialized to 0.5 (balanced prior)\n            self.gamma = nn.Parameter(torch.tensor([0.5]))\n        else:\n            self.gamma = None\n\n    def forward(self, guidance: torch.Tensor, cost_map: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Args:\n            guidance (I): Shallow backbone features of shape (B, C_in_guide, H_g, W_g)\n            cost_map (P): Upsampled semantic cost map of shape (B, C_in_cost, H_p, W_p)\n            \n        Returns:\n            Refined feature map of shape (B, C_in_cost, H_p, W_p)\n        \"\"\"\n        # Ensure spatial resolutions match\n        if guidance.shape[-2:] != cost_map.shape[-2:]:\n            guidance = F.interpolate(\n                guidance,\n                size=cost_map.shape[-2:],\n                mode=\"bilinear\",\n                align_corners=True\n            )\n\n        # Project guidance to appropriate channel representation\n        I_proj = self.guidance_proj(guidance)\n\n        # Apply closed-form guided filtering\n        q = self.filter_core(guidance=I_proj, cost_map=cost_map)\n\n        # Residual gating: q_final = cost_map + gamma * (q - cost_map)\n        if self.use_residual_gate:\n            q_final = cost_map + self.gamma * (q - cost_map)\n        else:\n            q_final = q\n\n        return q_final\n\n\nclass AGFM(nn.Module):\n    \"\"\"\n    Adaptive Guided Fusion Module (AGFM).\n    Combines Cross-Modal Guided Filter (CM-GF) with dual-branch residual feature stitching.\n    Acts as a drop-in replacement for standard concatenation-convolution blocks in SED.\n    \"\"\"\n\n    def __init__(\n        self,\n        in_cost_channels: int,\n        guidance_channels: int,\n        out_channels: int,\n        radius: int = 2,\n        eps: float = 1e-3,\n        mode: str = \"channelwise\"\n    ):\n        super().__init__()\n        self.cm_gf = CMGFModule(\n            guidance_channels=guidance_channels,\n            cost_map_channels=in_cost_channels,\n            radius=radius,\n            eps=eps,\n            mode=mode,\n            use_residual_gate=True\n        )\n\n        # Dual-branch fusion convolution (Pointwise + Depthwise-Separable for low FLOPs)\n        total_in = in_cost_channels + guidance_channels\n        self.fusion = nn.Sequential(\n            nn.Conv2d(total_in, out_channels, kernel_size=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, groups=out_channels, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n\n    def forward(self, cost_map: torch.Tensor, guidance: torch.Tensor) -> torch.Tensor:\n        \"\"\"\n        Args:\n            cost_map: Upsampled deep feature map (B, C_cost, H, W)\n            guidance: Shallow visual feature map (B, C_guide, H, W)\n        Returns:\n            Edge-sharpened fused representation (B, C_out, H, W)\n        \"\"\"\n        refined_cost = self.cm_gf(guidance=guidance, cost_map=cost_map)\n        if guidance.shape[-2:] != cost_map.shape[-2:]:\n            guidance = F.interpolate(\n                guidance,\n                size=cost_map.shape[-2:],\n                mode=\"bilinear\",\n                align_corners=True\n            )\n        fused = self.fusion(torch.cat([refined_cost, guidance], dim=1))\n        return fused\n")

# 2. Write updated config.py with USE_CM_GF support
with open('sed/config.py', 'w', encoding='utf-8') as f:
    f.write("# -*- coding: utf-8 -*-\n# Copyright (c) Facebook, Inc. and its affiliates.\nfrom detectron2.config import CfgNode as CN\n\n\ndef add_sed_config(cfg):\n    \"\"\"\n    Add config for MASK_FORMER.\n    \"\"\"\n    # data config\n    # select the dataset mapper\n    cfg.INPUT.DATASET_MAPPER_NAME = \"mask_former_semantic\"\n\n    cfg.DATASETS.VAL_ALL = (\"coco_2017_val_all_stuff_sem_seg\",)\n\n    # Color augmentation\n    cfg.INPUT.COLOR_AUG_SSD = False\n    # We retry random cropping until no single category in semantic segmentation GT occupies more\n    # than `SINGLE_CATEGORY_MAX_AREA` part of the crop.\n    cfg.INPUT.CROP.SINGLE_CATEGORY_MAX_AREA = 1.0\n    # Pad image and segmentation GT in dataset mapper.\n    cfg.INPUT.SIZE_DIVISIBILITY = -1\n\n    # solver config\n    # weight decay on embedding\n    cfg.SOLVER.WEIGHT_DECAY_EMBED = 0.0\n    # optimizer\n    cfg.SOLVER.OPTIMIZER = \"ADAMW\"\n    cfg.SOLVER.BACKBONE_MULTIPLIER = 0.1\n\n    # mask_former model config\n    cfg.MODEL.MASK_FORMER = CN()\n\n    # Sometimes `backbone.size_divisibility` is set to 0 for some backbone (e.g. ResNet)\n    # you can use this config to override\n    cfg.MODEL.MASK_FORMER.SIZE_DIVISIBILITY = 32\n\n    # swin transformer backbone\n    cfg.MODEL.SWIN = CN()\n    cfg.MODEL.SWIN.PRETRAIN_IMG_SIZE = 224\n    cfg.MODEL.SWIN.PATCH_SIZE = 4\n    cfg.MODEL.SWIN.EMBED_DIM = 96\n    cfg.MODEL.SWIN.DEPTHS = [2, 2, 6, 2]\n    cfg.MODEL.SWIN.NUM_HEADS = [3, 6, 12, 24]\n    cfg.MODEL.SWIN.WINDOW_SIZE = 7\n    cfg.MODEL.SWIN.MLP_RATIO = 4.0\n    cfg.MODEL.SWIN.QKV_BIAS = True\n    cfg.MODEL.SWIN.QK_SCALE = None\n    cfg.MODEL.SWIN.DROP_RATE = 0.0\n    cfg.MODEL.SWIN.ATTN_DROP_RATE = 0.0\n    cfg.MODEL.SWIN.DROP_PATH_RATE = 0.3\n    cfg.MODEL.SWIN.APE = False\n    cfg.MODEL.SWIN.PATCH_NORM = True\n    cfg.MODEL.SWIN.OUT_FEATURES = [\"res2\", \"res3\", \"res4\", \"res5\"]\n\n    # zero shot config\n    cfg.MODEL.SEM_SEG_HEAD.TRAIN_CLASS_JSON = \"datasets/ADE20K_2021_17_01/ADE20K_847.json\"\n    cfg.MODEL.SEM_SEG_HEAD.TEST_CLASS_JSON = \"datasets/ADE20K_2021_17_01/ADE20K_847.json\"\n    cfg.MODEL.SEM_SEG_HEAD.TRAIN_CLASS_INDEXES = \"datasets/coco/coco_stuff/split/seen_indexes.json\"\n    cfg.MODEL.SEM_SEG_HEAD.TEST_CLASS_INDEXES = \"datasets/coco/coco_stuff/split/unseen_indexes.json\"\n\n    cfg.MODEL.SEM_SEG_HEAD.CLIP_PRETRAINED = \"ViT-B/16\"\n\n    cfg.MODEL.PROMPT_ENSEMBLE = False\n    cfg.MODEL.PROMPT_ENSEMBLE_TYPE = \"single\"\n\n    cfg.MODEL.CLIP_PIXEL_MEAN = [122.7709383, 116.7460125, 104.09373615]\n    cfg.MODEL.CLIP_PIXEL_STD = [68.5005327, 66.6321579, 70.3231630]\n    # three styles for clip classification, crop, mask, cropmask\n\n    cfg.MODEL.SEM_SEG_HEAD.TEXT_GUIDANCE_DIM = 512\n    cfg.MODEL.SEM_SEG_HEAD.TEXT_GUIDANCE_PROJ_DIM = 128\n    cfg.MODEL.SEM_SEG_HEAD.APPEARANCE_GUIDANCE_DIM = 512\n    cfg.MODEL.SEM_SEG_HEAD.APPEARANCE_GUIDANCE_PROJ_DIM = 128\n\n    cfg.MODEL.SEM_SEG_HEAD.DECODER_DIMS = [64, 32]\n    cfg.MODEL.SEM_SEG_HEAD.DECODER_GUIDANCE_DIMS = [256, 128]\n    cfg.MODEL.SEM_SEG_HEAD.DECODER_GUIDANCE_PROJ_DIMS = [32, 16]\n\n    cfg.MODEL.SEM_SEG_HEAD.NUM_LAYERS = 4\n    cfg.MODEL.SEM_SEG_HEAD.NUM_HEADS = 4\n    cfg.MODEL.SEM_SEG_HEAD.HIDDEN_DIMS = 128\n    cfg.MODEL.SEM_SEG_HEAD.POOLING_SIZES = [6, 6]\n    cfg.MODEL.SEM_SEG_HEAD.FEATURE_RESOLUTION = [24, 24]\n    cfg.MODEL.SEM_SEG_HEAD.WINDOW_SIZES = 12\n    cfg.MODEL.SEM_SEG_HEAD.ATTENTION_TYPE = \"linear\"\n    cfg.MODEL.SEM_SEG_HEAD.CNEXT_TYPE = \"V1\"\n    cfg.MODEL.SEM_SEG_HEAD.KERNEL_SIZE = [7, 7, 7]\n\n    cfg.MODEL.SEM_SEG_HEAD.PROMPT_DEPTH = 0\n    cfg.MODEL.SEM_SEG_HEAD.PROMPT_LENGTH = 0\n    cfg.SOLVER.CLIP_MULTIPLIER = 0.01\n\n    cfg.MODEL.SEM_SEG_HEAD.CLIP_FINETUNE = \"attention\"\n    cfg.TEST.SLIDING_WINDOW = False\n    cfg.TEST.FAST_INFERENCE = False\n    cfg.TEST.TOPK = 1\n\n    # Cross-Modal Guided Filter (CM-GF) novelty config\n    cfg.MODEL.SEM_SEG_HEAD.USE_CM_GF = True\n    cfg.MODEL.SEM_SEG_HEAD.CM_GF_RADIUS = 2\n    cfg.MODEL.SEM_SEG_HEAD.CM_GF_EPS = 1e-3\n    cfg.MODEL.SEM_SEG_HEAD.CM_GF_MODE = \"channelwise\"\n    cfg.MODEL.SEM_SEG_HEAD.CM_GF_RESIDUAL = True\n\n    # convnext backbone\n    cfg.MODEL.ENC = CN()\n    cfg.MODEL.ENC.CLIP_MODEL_NAME = \"convnext_large_d_320\"\n    cfg.MODEL.ENC.CLIP_PRETRAINED_WEIGHTS = \"laion2b_s29b_b131k_ft_soup\"\n    cfg.MODEL.ENC.EMBED_DIM = 768\n")

# 3. Write updated sed_predictor.py with use_cm_gf support in __init__ and from_config
with open('sed/modeling/transformer/sed_predictor.py', 'w', encoding='utf-8') as f:
    f.write("# Copyright (c) Facebook, Inc. and its affiliates.\n# Modified by Bowen Cheng from: https://github.com/facebookresearch/detr/blob/master/models/detr.py\n# Modified by Jian Ding from: https://github.com/facebookresearch/MaskFormer/blob/main/mask_former/modeling/transformer/transformer_predictor.py\nimport fvcore.nn.weight_init as weight_init\nimport torch\n\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom detectron2.config import configurable\nfrom detectron2.layers import Conv2d\n\nfrom .model import Aggregator\nfrom sed.third_party import clip\nfrom sed.third_party import imagenet_templates\n\nimport numpy as np\nimport open_clip\nclass SEDPredictor(nn.Module):\n    @configurable\n    def __init__(\n        self,\n        *,\n        train_class_json: str,\n        test_class_json: str,\n        clip_pretrained: str,\n        prompt_ensemble_type: str,\n        text_guidance_dim: int,\n        text_guidance_proj_dim: int,\n        appearance_guidance_dim: int,\n        appearance_guidance_proj_dim: int,\n        prompt_depth: int,\n        prompt_length: int,\n        decoder_dims: list,\n        decoder_guidance_dims: list,\n        decoder_guidance_proj_dims: list,\n        num_heads: int,\n        num_layers: tuple,\n        hidden_dims: tuple,\n        pooling_sizes: tuple,\n        feature_resolution: tuple,\n        window_sizes: tuple,\n        attention_type: str,\n        cnext_type: str,\n        kernel_size: list,\n        fast_inference: bool,\n        topK: int,\n        use_cm_gf: bool = True,\n        cm_gf_radius: int = 2,\n        cm_gf_eps: float = 1e-3,\n        cm_gf_mode: str = \"channelwise\",\n        cm_gf_residual: bool = True,\n    ):\n        \"\"\"\n        Args:\n            \n        \"\"\"\n        super().__init__()\n        \n        import json\n        # use class_texts in train_forward, and test_class_texts in test_forward\n        with open(train_class_json, 'r') as f_in:\n            self.class_texts = json.load(f_in)\n        with open(test_class_json, 'r') as f_in:\n            self.test_class_texts = json.load(f_in)\n        assert self.class_texts != None\n        if self.test_class_texts == None:\n            self.test_class_texts = self.class_texts\n        device = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n  \n        self.tokenizer = None\n        if clip_pretrained == \"Convnext-L\":\n            name, pretrain = ('convnext_large_d_320', 'laion2b_s29b_b131k_ft_soup')\n            clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(\n                name,\n                pretrained=pretrain,\n                device=device,)\n\n            self.tokenizer = open_clip.get_tokenizer(name)\n        elif clip_pretrained == \"Convnext-B\":\n            name, pretrain = ('convnext_base_w_320', 'laion_aesthetic_s13b_b82k')\n            clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(\n                name,\n                pretrained=pretrain,\n                device=device,)\n\n            self.tokenizer = open_clip.get_tokenizer(name)\n        elif clip_pretrained == \"ViT-G\" or clip_pretrained == \"ViT-H\":\n            # for OpenCLIP models\n            name, pretrain = ('ViT-H-14', 'laion2b_s32b_b79k') if clip_pretrained == 'ViT-H' else ('ViT-bigG-14', 'laion2b_s39b_b160k')\n            clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(\n                name, \n                pretrained=pretrain, \n                device=device, \n                force_image_size=336,)\n        \n            self.tokenizer = open_clip.get_tokenizer(name)\n        else:\n            # for OpenAI models\n            clip_model, clip_preprocess = clip.load(clip_pretrained, device=device, jit=False, prompt_depth=prompt_depth, prompt_length=prompt_length)\n    \n        self.prompt_ensemble_type = prompt_ensemble_type        \n\n        if self.prompt_ensemble_type == \"imagenet_select\":\n            prompt_templates = imagenet_templates.IMAGENET_TEMPLATES_SELECT\n        elif self.prompt_ensemble_type == \"imagenet\":\n            prompt_templates = imagenet_templates.IMAGENET_TEMPLATES\n        elif self.prompt_ensemble_type == \"single\":\n            prompt_templates = ['A photo of a {} in the scene',]\n        else:\n            raise NotImplementedError\n\n        self.text_features = self.class_embeddings(self.class_texts, prompt_templates, clip_model).permute(1, 0, 2).float()\n        self.text_features_test = self.class_embeddings(self.test_class_texts, prompt_templates, clip_model).permute(1, 0, 2).float()\n        \n        self.clip_model = clip_model.float()\n        self.clip_preprocess = clip_preprocess\n        \n        transformer = Aggregator(\n            text_guidance_dim=text_guidance_dim,\n            text_guidance_proj_dim=text_guidance_proj_dim,\n            appearance_guidance_dim=appearance_guidance_dim,\n            appearance_guidance_proj_dim=appearance_guidance_proj_dim,\n            decoder_dims=decoder_dims,\n            decoder_guidance_dims=decoder_guidance_dims,\n            decoder_guidance_proj_dims=decoder_guidance_proj_dims,\n            num_layers=num_layers,\n            nheads=num_heads, \n            hidden_dim=hidden_dims,\n            pooling_size=pooling_sizes,\n            feature_resolution=feature_resolution,\n            window_size=window_sizes,\n            attention_type=attention_type,\n            prompt_channel=len(prompt_templates),\n            cnext_type=cnext_type,\n            kernel_size=kernel_size,\n            fast_inference=fast_inference,\n            topK=topK,\n            use_cm_gf=use_cm_gf,\n            cm_gf_radius=cm_gf_radius,\n            cm_gf_eps=cm_gf_eps,\n            cm_gf_mode=cm_gf_mode,\n            cm_gf_residual=cm_gf_residual,\n            )\n        self.transformer = transformer\n\n    @classmethod\n    def from_config(cls, cfg):#, in_channels, mask_classification):\n        ret = {}\n\n        ret[\"train_class_json\"] = cfg.MODEL.SEM_SEG_HEAD.TRAIN_CLASS_JSON\n        ret[\"test_class_json\"] = cfg.MODEL.SEM_SEG_HEAD.TEST_CLASS_JSON\n        ret[\"clip_pretrained\"] = cfg.MODEL.SEM_SEG_HEAD.CLIP_PRETRAINED\n        ret[\"prompt_ensemble_type\"] = cfg.MODEL.PROMPT_ENSEMBLE_TYPE\n\n        # Aggregator parameters:\n        ret[\"text_guidance_dim\"] = cfg.MODEL.SEM_SEG_HEAD.TEXT_GUIDANCE_DIM\n        ret[\"text_guidance_proj_dim\"] = cfg.MODEL.SEM_SEG_HEAD.TEXT_GUIDANCE_PROJ_DIM\n        ret[\"appearance_guidance_dim\"] = cfg.MODEL.SEM_SEG_HEAD.APPEARANCE_GUIDANCE_DIM\n        ret[\"appearance_guidance_proj_dim\"] = cfg.MODEL.SEM_SEG_HEAD.APPEARANCE_GUIDANCE_PROJ_DIM\n\n        ret[\"decoder_dims\"] = cfg.MODEL.SEM_SEG_HEAD.DECODER_DIMS\n        ret[\"decoder_guidance_dims\"] = cfg.MODEL.SEM_SEG_HEAD.DECODER_GUIDANCE_DIMS\n        ret[\"decoder_guidance_proj_dims\"] = cfg.MODEL.SEM_SEG_HEAD.DECODER_GUIDANCE_PROJ_DIMS\n\n        ret[\"prompt_depth\"] = cfg.MODEL.SEM_SEG_HEAD.PROMPT_DEPTH\n        ret[\"prompt_length\"] = cfg.MODEL.SEM_SEG_HEAD.PROMPT_LENGTH\n\n        ret[\"num_layers\"] = cfg.MODEL.SEM_SEG_HEAD.NUM_LAYERS\n        ret[\"num_heads\"] = cfg.MODEL.SEM_SEG_HEAD.NUM_HEADS\n        ret[\"hidden_dims\"] = cfg.MODEL.SEM_SEG_HEAD.HIDDEN_DIMS\n        ret[\"pooling_sizes\"] = cfg.MODEL.SEM_SEG_HEAD.POOLING_SIZES\n        ret[\"feature_resolution\"] = cfg.MODEL.SEM_SEG_HEAD.FEATURE_RESOLUTION\n        ret[\"window_sizes\"] = cfg.MODEL.SEM_SEG_HEAD.WINDOW_SIZES\n        ret[\"attention_type\"] = cfg.MODEL.SEM_SEG_HEAD.ATTENTION_TYPE\n        ret[\"cnext_type\"] = cfg.MODEL.SEM_SEG_HEAD.CNEXT_TYPE\n        ret[\"kernel_size\"] = cfg.MODEL.SEM_SEG_HEAD.KERNEL_SIZE\n        ret[\"fast_inference\"] = cfg.TEST.FAST_INFERENCE\n        ret[\"topK\"] = cfg.TEST.TOPK\n\n        # Cross-Modal Guided Filter (CM-GF) parameters\n        ret[\"use_cm_gf\"] = getattr(cfg.MODEL.SEM_SEG_HEAD, \"USE_CM_GF\", True)\n        ret[\"cm_gf_radius\"] = getattr(cfg.MODEL.SEM_SEG_HEAD, \"CM_GF_RADIUS\", 2)\n        ret[\"cm_gf_eps\"] = getattr(cfg.MODEL.SEM_SEG_HEAD, \"CM_GF_EPS\", 1e-3)\n        ret[\"cm_gf_mode\"] = getattr(cfg.MODEL.SEM_SEG_HEAD, \"CM_GF_MODE\", \"channelwise\")\n        ret[\"cm_gf_residual\"] = getattr(cfg.MODEL.SEM_SEG_HEAD, \"CM_GF_RESIDUAL\", True)\n\n        return ret\n\n    def forward(self, x, vis_guidance, ):\n        logit_scale = self.clip_model.logit_scale\n        vis = [vis_guidance[k] for k in vis_guidance.keys()][::-1]\n        text = self.text_features if self.training else self.text_features_test\n        text = text.repeat(x.shape[0], 1, 1, 1)\n        out = self.transformer(x, text, vis, logit_scale)\n        return out\n\n    @torch.no_grad()\n    def class_embeddings(self, classnames, templates, clip_model):\n        zeroshot_weights = []\n        for classname in classnames:\n            if ', ' in classname:\n                classname_splits = classname.split(', ')\n                texts = []\n                for template in templates:\n                    for cls_split in classname_splits:\n                        texts.append(template.format(cls_split))\n            else:\n                texts = [template.format(classname) for template in templates]  # format with class\n            if self.tokenizer is not None:\n                texts = self.tokenizer(texts).cuda()\n            else: \n                texts = clip.tokenize(texts).cuda()\n            class_embeddings = clip_model.encode_text(texts)\n            class_embeddings /= class_embeddings.norm(dim=-1, keepdim=True)\n            if len(templates) != class_embeddings.shape[0]:\n                class_embeddings = class_embeddings.reshape(len(templates), -1, class_embeddings.shape[-1]).mean(dim=1)\n                class_embeddings /= class_embeddings.norm(dim=-1, keepdim=True)\n            class_embedding = class_embeddings\n            zeroshot_weights.append(class_embedding)\n        zeroshot_weights = torch.stack(zeroshot_weights, dim=1).cuda()\n        return zeroshot_weights")

# 4. Write updated model.py with CM-GF inside Up and Aggregator
with open('sed/modeling/transformer/model.py', 'w', encoding='utf-8') as f:
    f.write("import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom einops import rearrange, repeat\nfrom einops.layers.torch import Rearrange\nfrom .convnext import ConvNextBlock, ConvNextV2Block\nfrom .cm_gf import CMGFModule\nfrom timm.layers import PatchEmbed, Mlp, DropPath, to_2tuple, to_ntuple, trunc_normal_, _assert\n\ndef window_partition(x, window_size: int):\n    \"\"\"\n    Args:\n        x: (B, H, W, C)\n        window_size (int): window size\n\n    Returns:\n        windows: (num_windows*B, window_size, window_size, C)\n    \"\"\"\n    B, H, W, C = x.shape\n    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)\n    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)\n    return windows\n\n\ndef window_reverse(windows, window_size: int, H: int, W: int):\n    \"\"\"\n    Args:\n        windows: (num_windows*B, window_size, window_size, C)\n        window_size (int): Window size\n        H (int): Height of image\n        W (int): Width of image\n\n    Returns:\n        x: (B, H, W, C)\n    \"\"\"\n    B = int(windows.shape[0] / (H * W / window_size / window_size))\n    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)\n    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)\n    return x\n\n\n\nclass WindowAttention(nn.Module):\n    r\"\"\" Window based multi-head self attention (W-MSA) module with relative position bias.\n    It supports both of shifted and non-shifted window.\n\n    Args:\n        dim (int): Number of input channels.\n        num_heads (int): Number of attention heads.\n        head_dim (int): Number of channels per head (dim // num_heads if not set)\n        window_size (tuple[int]): The height and width of the window.\n        qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True\n        attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0\n        proj_drop (float, optional): Dropout ratio of output. Default: 0.0\n    \"\"\"\n\n    def __init__(self, dim, appearance_guidance_dim, num_heads, head_dim=None, window_size=7, qkv_bias=True, attn_drop=0., proj_drop=0.):\n\n        super().__init__()\n        self.dim = dim\n        self.window_size = to_2tuple(window_size)  # Wh, Ww\n        win_h, win_w = self.window_size\n        self.window_area = win_h * win_w\n        self.num_heads = num_heads\n        head_dim = head_dim or dim // num_heads\n        attn_dim = head_dim * num_heads\n        self.scale = head_dim ** -0.5\n\n        self.q = nn.Linear(dim + appearance_guidance_dim, attn_dim, bias=qkv_bias)\n        self.k = nn.Linear(dim + appearance_guidance_dim, attn_dim, bias=qkv_bias)\n        self.v = nn.Linear(dim, attn_dim, bias=qkv_bias)\n        self.attn_drop = nn.Dropout(attn_drop)\n        self.proj = nn.Linear(attn_dim, dim)\n        self.proj_drop = nn.Dropout(proj_drop)\n\n        self.softmax = nn.Softmax(dim=-1)\n\n    def forward(self, x, mask=None):\n        \"\"\"\n        Args:\n            x: input features with shape of (num_windows*B, N, C)\n            mask: (0/-inf) mask with shape of (num_windows, Wh*Ww, Wh*Ww) or None\n        \"\"\"\n        B_, N, C = x.shape\n        \n        q = self.q(x).reshape(B_, N, self.num_heads, -1).permute(0, 2, 1, 3)\n        k = self.k(x).reshape(B_, N, self.num_heads, -1).permute(0, 2, 1, 3)\n        v = self.v(x[:, :, :self.dim]).reshape(B_, N, self.num_heads, -1).permute(0, 2, 1, 3)\n\n        q = q * self.scale\n        attn = (q @ k.transpose(-2, -1))\n\n        if mask is not None:\n            num_win = mask.shape[0]\n            attn = attn.view(B_ // num_win, num_win, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)\n            attn = attn.view(-1, self.num_heads, N, N)\n            attn = self.softmax(attn)\n        else:\n            attn = self.softmax(attn)\n\n        attn = self.attn_drop(attn)\n\n        x = (attn @ v).transpose(1, 2).reshape(B_, N, -1)\n        x = self.proj(x)\n        x = self.proj_drop(x)\n        return x\n\n\nclass SwinTransformerBlock(nn.Module):\n    r\"\"\" Swin Transformer Block.\n\n    Args:\n        dim (int): Number of input channels.\n        input_resolution (tuple[int]): Input resulotion.\n        window_size (int): Window size.\n        num_heads (int): Number of attention heads.\n        head_dim (int): Enforce the number of channels per head\n        shift_size (int): Shift size for SW-MSA.\n        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.\n        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True\n        drop (float, optional): Dropout rate. Default: 0.0\n        attn_drop (float, optional): Attention dropout rate. Default: 0.0\n        drop_path (float, optional): Stochastic depth rate. Default: 0.0\n        act_layer (nn.Module, optional): Activation layer. Default: nn.GELU\n        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm\n    \"\"\"\n\n    def __init__(\n            self, dim, appearance_guidance_dim, input_resolution, num_heads=4, head_dim=None, window_size=7, shift_size=0,\n            mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0., drop_path=0.,\n            act_layer=nn.GELU, norm_layer=nn.LayerNorm):\n        super().__init__()\n        self.dim = dim\n        self.input_resolution = input_resolution\n        self.window_size = window_size\n        self.shift_size = shift_size\n        self.mlp_ratio = mlp_ratio\n        if min(self.input_resolution) <= self.window_size:\n            # if window size is larger than input resolution, we don't partition windows\n            self.shift_size = 0\n            self.window_size = min(self.input_resolution)\n        assert 0 <= self.shift_size < self.window_size, \"shift_size must in 0-window_size\"\n\n        self.norm1 = norm_layer(dim)\n        self.attn = WindowAttention(\n            dim, appearance_guidance_dim=appearance_guidance_dim, num_heads=num_heads, head_dim=head_dim, window_size=to_2tuple(self.window_size),\n            qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)\n\n        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()\n        self.norm2 = norm_layer(dim)\n        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), act_layer=act_layer, drop=drop)\n\n        if self.shift_size > 0:\n            # calculate attention mask for SW-MSA\n            H, W = self.input_resolution\n            img_mask = torch.zeros((1, H, W, 1))  # 1 H W 1\n            cnt = 0\n            for h in (\n                    slice(0, -self.window_size),\n                    slice(-self.window_size, -self.shift_size),\n                    slice(-self.shift_size, None)):\n                for w in (\n                        slice(0, -self.window_size),\n                        slice(-self.window_size, -self.shift_size),\n                        slice(-self.shift_size, None)):\n                    img_mask[:, h, w, :] = cnt\n                    cnt += 1\n            mask_windows = window_partition(img_mask, self.window_size)  # num_win, window_size, window_size, 1\n            mask_windows = mask_windows.view(-1, self.window_size * self.window_size)\n            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)\n            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))\n        else:\n            attn_mask = None\n\n        self.register_buffer(\"attn_mask\", attn_mask)\n\n    def forward(self, x, appearance_guidance):\n        H, W = self.input_resolution\n        B, L, C = x.shape\n        assert L == H * W, \"input feature has wrong size\"\n\n        shortcut = x\n        x = self.norm1(x)\n        x = x.view(B, H, W, C)\n        if appearance_guidance is not None:\n            appearance_guidance = appearance_guidance.view(B, H, W, -1)\n            x = torch.cat([x, appearance_guidance], dim=-1)\n\n        # cyclic shift\n        if self.shift_size > 0:\n            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))\n        else:\n            shifted_x = x\n\n        # partition windows\n        x_windows = window_partition(shifted_x, self.window_size)  # num_win*B, window_size, window_size, C\n        x_windows = x_windows.view(-1, self.window_size * self.window_size, x_windows.shape[-1])  # num_win*B, window_size*window_size, C\n\n        # W-MSA/SW-MSA\n        attn_windows = self.attn(x_windows, mask=self.attn_mask)  # num_win*B, window_size*window_size, C\n\n        # merge windows\n        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)\n        shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C\n\n        # reverse cyclic shift\n        if self.shift_size > 0:\n            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))\n        else:\n            x = shifted_x\n        x = x.view(B, H * W, C)\n\n        # FFN\n        x = shortcut + self.drop_path(x)\n        x = x + self.drop_path(self.mlp(self.norm2(x)))\n\n        return x\n\n\nclass SwinTransformerBlockWrapper(nn.Module):\n    def __init__(self, dim, appearance_guidance_dim, input_resolution, nheads=4, window_size=5):\n        super().__init__()\n        self.block_1 = SwinTransformerBlock(dim, appearance_guidance_dim, input_resolution, num_heads=nheads, head_dim=None, window_size=window_size, shift_size=0)\n        self.block_2 = SwinTransformerBlock(dim, appearance_guidance_dim, input_resolution, num_heads=nheads, head_dim=None, window_size=window_size, shift_size=window_size // 2)\n        self.guidance_norm = nn.LayerNorm(appearance_guidance_dim) if appearance_guidance_dim > 0 else None\n    \n    def forward(self, x, appearance_guidance=None):\n        \"\"\"\n        Arguments:\n            x: B C T H W\n            appearance_guidance: B C H W\n        \"\"\"\n        BT, C, H, W = x.shape\n        x = rearrange(x, 'BT C H W -> BT (H W) C')\n        if appearance_guidance is not None:\n            # appearance_guidance = self.guidance_norm(repeat(appearance_guidance, 'B C H W -> (B T) (H W) C', T=T))\n            pass\n        x = self.block_1(x, appearance_guidance)\n        x = self.block_2(x, appearance_guidance)\n        x = rearrange(x, 'BT (H W) C -> BT C H W', H=H, W=W)\n        return x\n\n\ndef elu_feature_map(x):\n    return torch.nn.functional.elu(x) + 1\n\n\nclass LinearAttention(nn.Module):\n    def __init__(self, eps=1e-6):\n        super().__init__()\n        self.feature_map = elu_feature_map\n        self.eps = eps\n\n    def forward(self, queries, keys, values):\n        \"\"\" Multi-Head linear attention proposed in \"Transformers are RNNs\"\n        Args:\n            queries: [N, L, H, D]\n            keys: [N, S, H, D]\n            values: [N, S, H, D]\n            q_mask: [N, L]\n            kv_mask: [N, S]\n        Returns:\n            queried_values: (N, L, H, D)\n        \"\"\"\n        Q = self.feature_map(queries)\n        K = self.feature_map(keys)\n\n        v_length = values.size(1)\n        values = values / v_length  # prevent fp16 overflow\n        KV = torch.einsum(\"nshd,nshv->nhdv\", K, values)  # (S,D)' @ S,V\n        Z = 1 / (torch.einsum(\"nlhd,nhd->nlh\", Q, K.sum(dim=1)) + self.eps)\n        queried_values = torch.einsum(\"nlhd,nhdv,nlh->nlhv\", Q, KV, Z) * v_length\n\n        return queried_values.contiguous()\n\n\nclass FullAttention(nn.Module):\n    def __init__(self, use_dropout=False, attention_dropout=0.1):\n        super().__init__()\n        self.use_dropout = use_dropout\n        self.dropout = nn.Dropout(attention_dropout)\n\n    def forward(self, queries, keys, values, q_mask=None, kv_mask=None):\n        \"\"\" Multi-head scaled dot-product attention, a.k.a full attention.\n        Args:\n            queries: [N, L, H, D]\n            keys: [N, S, H, D]\n            values: [N, S, H, D]\n            q_mask: [N, L]\n            kv_mask: [N, S]\n        Returns:\n            queried_values: (N, L, H, D)\n        \"\"\"\n\n        # Compute the unnormalized attention and apply the masks\n        QK = torch.einsum(\"nlhd,nshd->nlsh\", queries, keys)\n        if kv_mask is not None:\n            QK.masked_fill_(~(q_mask[:, :, None, None] * kv_mask[:, None, :, None]), float('-inf'))\n\n        # Compute the attention and the weighted average\n        softmax_temp = 1. / queries.size(3)**.5  # sqrt(D)\n        A = torch.softmax(softmax_temp * QK, dim=2)\n        if self.use_dropout:\n            A = self.dropout(A)\n\n        queried_values = torch.einsum(\"nlsh,nshd->nlhd\", A, values)\n\n        return queried_values.contiguous()\n\n\nclass AttentionLayer(nn.Module):\n    def __init__(self, hidden_dim, guidance_dim, nheads=8, attention_type='linear'):\n        super().__init__()\n        self.nheads = nheads\n        self.q = nn.Linear(hidden_dim + guidance_dim, hidden_dim)\n        self.k = nn.Linear(hidden_dim + guidance_dim, hidden_dim)\n        self.v = nn.Linear(hidden_dim, hidden_dim)\n\n        if attention_type == 'linear':\n            self.attention = LinearAttention()\n        elif attention_type == 'full':\n            self.attention = FullAttention()\n        else:\n            raise NotImplementedError\n    \n    def forward(self, x, guidance):\n        \"\"\"\n        Arguments:\n            x: B, L, C\n            guidance: B, L, C\n        \"\"\"\n        q = self.q(torch.cat([x, guidance], dim=-1)) if guidance is not None else self.q(x)\n        k = self.k(torch.cat([x, guidance], dim=-1)) if guidance is not None else self.k(x)\n        v = self.v(x)\n\n        q = rearrange(q, 'B L (H D) -> B L H D', H=self.nheads)\n        k = rearrange(k, 'B S (H D) -> B S H D', H=self.nheads)\n        v = rearrange(v, 'B S (H D) -> B S H D', H=self.nheads)\n\n        out = self.attention(q, k, v)\n        out = rearrange(out, 'B L H D -> B L (H D)')\n        return out\n\n\nclass ClassTransformerLayer(nn.Module):\n    def __init__(self, hidden_dim=64, guidance_dim=64, nheads=8, attention_type='linear', pooling_size=(4, 4)) -> None:\n        super().__init__()\n        self.pool = nn.AvgPool2d(pooling_size)\n        self.attention = AttentionLayer(hidden_dim, guidance_dim, nheads=nheads, attention_type=attention_type)\n        self.MLP = nn.Sequential(\n            nn.Linear(hidden_dim, hidden_dim * 4),\n            nn.ReLU(),\n            nn.Linear(hidden_dim * 4, hidden_dim)\n        )\n\n        self.norm1 = nn.LayerNorm(hidden_dim)\n        self.norm2 = nn.LayerNorm(hidden_dim)\n    \n    def pool_features(self, x):\n        \"\"\"\n        Intermediate pooling layer for computational efficiency.\n        Arguments:\n            x: B, C, T, H, W\n        \"\"\"\n        B = x.size(0)\n        x = rearrange(x, 'B C T H W -> (B T) C H W')\n        x = self.pool(x)\n        x = rearrange(x, '(B T) C H W -> B C T H W', B=B)\n        return x\n\n    def forward(self, x, guidance):\n        \"\"\"\n        Arguments:\n            x: B, C, T, H, W\n            guidance: B, T, C\n        \"\"\"\n        B, _, _, H, W = x.size()\n        x_pool = self.pool_features(x)\n        *_, H_pool, W_pool = x_pool.size()\n\n        x_pool = rearrange(x_pool, 'B C T H W -> (B H W) T C')\n        if guidance is not None:\n            guidance = repeat(guidance, 'B T C -> (B H W) T C', H=H_pool, W=W_pool)\n\n        x_pool = x_pool + self.attention(self.norm1(x_pool), guidance) # Attention\n        x_pool = x_pool + self.MLP(self.norm2(x_pool)) # MLP\n\n        x_pool = rearrange(x_pool, '(B H W) T C -> (B T) C H W', H=H_pool, W=W_pool)\n        x_pool = F.interpolate(x_pool, size=(H, W), mode='bilinear', align_corners=True)\n        x_pool = rearrange(x_pool, '(B T) C H W -> B C T H W', B=B)\n\n        x = x + x_pool # Residual\n        return x\n\n\ndef conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):\n    \"\"\"3x3 convolution with padding\"\"\"\n    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,\n                     padding=dilation, groups=groups, bias=False, dilation=dilation)\n\n\ndef conv1x1(in_planes, out_planes, stride=1):\n    \"\"\"1x1 convolution\"\"\"\n    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)\n\n\nclass Bottleneck(nn.Module):\n    expansion = 4\n    __constants__ = ['downsample']\n\n    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,\n                 base_width=64, dilation=1, norm_layer=None):\n        super(Bottleneck, self).__init__()\n        if norm_layer is None:\n            norm_layer = nn.BatchNorm2d\n        width = int(planes * (base_width / 64.)) * groups\n        # Both self.conv2 and self.downsample layers downsample the input when stride != 1\n        self.conv1 = conv1x1(inplanes, width)\n        self.bn1 = norm_layer(width)\n        self.conv2 = conv3x3(width, width, stride, groups, dilation)\n        self.bn2 = norm_layer(width)\n        self.conv3 = conv1x1(width, planes * self.expansion)\n        self.bn3 = norm_layer(planes * self.expansion)\n        self.relu = nn.ReLU(inplace=True)\n        self.downsample = downsample\n        self.stride = stride\n\n    def forward(self, x):\n        identity = x\n\n        out = self.conv1(x)\n        out = self.bn1(out)\n        out = self.relu(out)\n\n        out = self.conv2(out)\n        out = self.bn2(out)\n        out = self.relu(out)\n\n        out = self.conv3(out)\n        out = self.bn3(out)\n\n        if self.downsample is not None:\n            identity = self.downsample(x)\n\n        out += identity\n        out = self.relu(out)\n\n        return out\n\n\nclass AggregatorLayer(nn.Module):\n    def __init__(self, hidden_dim=64, text_guidance_dim=512, appearance_guidance=512, nheads=4, input_resolution=(20, 20), pooling_size=(5, 5), window_size=(10, 10), attention_type='linear') -> None:\n        super().__init__()\n        self.swin_block = SwinTransformerBlockWrapper(hidden_dim, appearance_guidance, input_resolution, nheads, window_size)\n        self.attention = ClassTransformerLayer(hidden_dim, text_guidance_dim, nheads=nheads, attention_type=attention_type, pooling_size=pooling_size)\n\n\n    def forward(self, x, appearance_guidance, text_guidance):\n        \"\"\"\n        Arguments:\n            x: B C T H W\n        \"\"\"\n        x = self.swin_block(x, appearance_guidance)\n        x = self.attention(x, text_guidance)\n        return x\n\n\nclass AggregatorResNetLayer(nn.Module):\n    def __init__(self, hidden_dim=64, appearance_guidance=512) -> None:\n        super().__init__()\n        self.conv_linear = nn.Conv2d(hidden_dim + appearance_guidance, hidden_dim, kernel_size=1, stride=1)\n        self.conv_layer = Bottleneck(hidden_dim, hidden_dim // 4)\n\n\n    def forward(self, x, appearance_guidance):\n        \"\"\"\n        Arguments:\n            x: B C T H W\n        \"\"\"\n        B, T = x.size(0), x.size(2)\n        x = rearrange(x, 'B C T H W -> (B T) C H W')\n        appearance_guidance = repeat(appearance_guidance, 'B C H W -> (B T) C H W', T=T)\n\n        x = self.conv_linear(torch.cat([x, appearance_guidance], dim=1))\n        x = self.conv_layer(x)\n        x = rearrange(x, '(B T) C H W -> B C T H W', B=B)\n        return x\n\n\nclass DoubleConv(nn.Module):\n    \"\"\"(convolution => [GN] => ReLU) * 2\"\"\"\n\n    def __init__(self, in_channels, out_channels, mid_channels=None):\n        super().__init__()\n        if not mid_channels:\n            mid_channels = out_channels\n        self.double_conv = nn.Sequential(\n            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),\n            nn.GroupNorm(mid_channels // 16, mid_channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),\n            nn.GroupNorm(mid_channels // 16, mid_channels),\n            nn.ReLU(inplace=True)\n        )\n\n    def forward(self, x):\n        return self.double_conv(x)\n\n\nclass Up(nn.Module):\n    \"\"\"Upscaling then double conv with optional Cross-Modal Guided Filter (CM-GF)\"\"\"\n\n    def __init__(self, in_channels, out_channels, guidance_channels, text_guidance_channels, nheads, attention_type, pooling_size, cnext_type=\"V1\", kernel_size=7,\n                 input_resolution=(24,24), window_size=(10, 10),\n                 use_cm_gf=True, cm_gf_radius=2, cm_gf_eps=1e-3, cm_gf_mode=\"channelwise\", cm_gf_residual=True):\n        super().__init__()\n        corr_guidance_channels = guidance_channels\n        self.up = nn.ConvTranspose2d(in_channels, in_channels - guidance_channels, kernel_size=2, stride=2)\n        self.conv = DoubleConv(in_channels+corr_guidance_channels, out_channels)\n        if cnext_type==\"V1\":\n            self.cnext = ConvNextBlock(in_channels, kernel_size, 0.0, 1.0)\n        elif cnext_type==\"V2\":\n            self.cnext = ConvNextV2Block(in_channels, kernel_size, 0.0)\n        elif cnext_type == \"Swin\":\n            self.cnext = SwinTransformerBlockWrapper(in_channels, 0, input_resolution, nheads, window_size)\n        else:\n            raise NotImplementedError\n        self.attention = ClassTransformerLayer(in_channels, text_guidance_channels, nheads=nheads, attention_type=attention_type, pooling_size=pooling_size)\n\n        self.use_cm_gf = use_cm_gf\n        if self.use_cm_gf:\n            self.cm_gf = CMGFModule(\n                guidance_channels=guidance_channels,\n                cost_map_channels=in_channels - guidance_channels,\n                radius=cm_gf_radius,\n                eps=cm_gf_eps,\n                mode=cm_gf_mode,\n                use_residual_gate=cm_gf_residual\n            )\n        else:\n            self.cm_gf = None\n\n    def forward(self, x, text_guidance, guidance=None, corr_guidance=None):\n        B, _, _, _ = guidance.size()\n        x = self.cnext(x)\n        x = rearrange(x, '(B T) C H W -> B C T H W', B=B)\n        x = self.attention(x, text_guidance)\n        x = rearrange(x, 'B C T H W -> (B T) C H W')\n        x = self.up(x)\n        if guidance is not None:\n            T = x.size(0) // guidance.size(0)\n            guidance = repeat(guidance, \"B C H W -> (B T) C H W\", T=T)\n            if self.use_cm_gf and self.cm_gf is not None:\n                # CM-GF: Refine upsampled coarse semantic features using shallow visual guidance\n                x_refined = self.cm_gf(guidance=guidance, cost_map=x)\n                x = torch.cat([x_refined, guidance], dim=1)\n            else:\n                x = torch.cat([x, guidance], dim=1)\n        if corr_guidance is not None:\n            x = torch.cat([x, corr_guidance], dim=1)\n        return self.conv(x)\n\n\nclass Aggregator(nn.Module):\n    def __init__(self, \n        text_guidance_dim=512,\n        text_guidance_proj_dim=128,\n        appearance_guidance_dim=512,\n        appearance_guidance_proj_dim=128,\n        decoder_dims = (64, 32),\n        decoder_guidance_dims=(256, 128),\n        decoder_corr_guidance_dims=(80, 32, 16),\n        decoder_guidance_proj_dims=(32, 16),\n        num_layers=4, \n        nheads=4, \n        hidden_dim=128,\n        pooling_size=(6, 6),\n        feature_resolution=(24, 24),\n        window_size=12,\n        attention_type='linear',\n        prompt_channel=80,\n        cnext_type=\"V1\",\n        kernel_size=(7, 7, 7),\n        fast_inference=False,\n        topK=1,\n        use_cm_gf=True,\n        cm_gf_radius=2,\n        cm_gf_eps=1e-3,\n        cm_gf_mode=\"channelwise\",\n        cm_gf_residual=True,\n    ) -> None:\n        super().__init__()\n        self.num_layers = num_layers\n        self.hidden_dim = hidden_dim\n\n        # self.layers = nn.ModuleList([\n        #     AggregatorLayer(\n        #         hidden_dim=hidden_dim, text_guidance_dim=text_guidance_proj_dim, appearance_guidance=appearance_guidance_proj_dim,\n        #         nheads=nheads, input_resolution=feature_resolution, pooling_size=pooling_size, window_size=window_size, attention_type=attention_type\n        #     ) for _ in range(num_layers)\n        # ])\n\n        self.conv1 = nn.Conv2d(prompt_channel, hidden_dim, kernel_size=7, stride=1, padding=3)\n\n        # self.guidance_projection = nn.Sequential(\n        #     nn.Conv2d(appearance_guidance_dim, appearance_guidance_proj_dim, kernel_size=3, stride=1, padding=1),\n        #     nn.ReLU(),\n        # ) if appearance_guidance_dim > 0 else None\n\n        self.text_guidance_projection = nn.ModuleList([\n            nn.Sequential(\n                nn.Linear(text_guidance_dim, tp),\n                nn.ReLU(),\n            ) for tp in [hidden_dim, decoder_dims[0], decoder_dims[1]]\n        ]) if text_guidance_dim > 0 else None\n\n        if text_guidance_dim > 0:\n            text_guidance_channels = [hidden_dim, decoder_dims[0], decoder_dims[1]]\n        else:\n            text_guidance_channels = [0, 0, 0]\n\n        self.decoder_guidance_projection = nn.ModuleList([\n            nn.Sequential(\n                nn.Conv2d(d, dp, kernel_size=3, stride=1, padding=1),\n                nn.ReLU(),\n            ) for d, dp in zip(decoder_guidance_dims, decoder_guidance_proj_dims)\n        ]) if decoder_guidance_dims[0] > 0 else None\n\n        self.decoder_corr_guidance_projection = nn.ModuleList([\n            nn.Sequential(\n                nn.ConvTranspose2d(d, dp, kernel_size=2, stride=2),\n                nn.ReLU(),\n            ) for d, dp in zip(decoder_corr_guidance_dims, decoder_guidance_proj_dims)\n        ]) if decoder_corr_guidance_dims[0] > 0 else None\n\n        self.decoder1 = Up(hidden_dim, decoder_dims[0], decoder_guidance_proj_dims[0], text_guidance_channels[0], nheads, attention_type, pooling_size, cnext_type, kernel_size[0], (24, 24), window_size,\n                           use_cm_gf=use_cm_gf, cm_gf_radius=cm_gf_radius, cm_gf_eps=cm_gf_eps, cm_gf_mode=cm_gf_mode, cm_gf_residual=cm_gf_residual)\n        self.decoder2 = Up(decoder_dims[0], decoder_dims[1], decoder_guidance_proj_dims[1], text_guidance_channels[1], nheads, attention_type, pooling_size, cnext_type, kernel_size[1], (48, 48), window_size,\n                           use_cm_gf=use_cm_gf, cm_gf_radius=cm_gf_radius, cm_gf_eps=cm_gf_eps, cm_gf_mode=cm_gf_mode, cm_gf_residual=cm_gf_residual)\n        self.decoder3 = Up(decoder_dims[1], decoder_dims[2], decoder_guidance_proj_dims[2], text_guidance_channels[2], nheads, attention_type, pooling_size, cnext_type, kernel_size[2], (96, 96), window_size,\n                           use_cm_gf=use_cm_gf, cm_gf_radius=cm_gf_radius, cm_gf_eps=cm_gf_eps, cm_gf_mode=cm_gf_mode, cm_gf_residual=cm_gf_residual)\n        self.head0 = nn.Conv2d(hidden_dim, 1, kernel_size=3, stride=1, padding=1)\n        self.head1 = nn.Conv2d(decoder_dims[0], 1, kernel_size=3, stride=1, padding=1)\n        self.head2 = nn.Conv2d(decoder_dims[1], 1, kernel_size=3, stride=1, padding=1)\n        self.head = nn.Conv2d(decoder_dims[2], 1, kernel_size=3, stride=1, padding=1)\n        self.fast_infer = fast_inference\n        self.topK = topK\n        # self.clip_vis_projection = nn.Conv2d(1536, 768, kernel_size=3, stride=1, padding=1)\n\n    def feature_map(self, img_feats, text_feats):\n        img_feats = F.normalize(img_feats, dim=1) # B C H W\n        img_feats = repeat(img_feats, \"B C H W -> B C T H W\", T=text_feats.shape[1])\n        text_feats = F.normalize(text_feats, dim=-1) # B T P C\n        text_feats = text_feats.mean(dim=-2)\n        text_feats = F.normalize(text_feats, dim=-1) # B T C\n        text_feats = repeat(text_feats, \"B T C -> B C T H W\", H=img_feats.shape[-2], W=img_feats.shape[-1])\n        return torch.cat((img_feats, text_feats), dim=1) # B 2C T H W\n\n    def correlation(self, img_feats, text_feats, logit_scale):\n        img_feats = F.normalize(img_feats, dim=1) # B C H W\n        text_feats = F.normalize(text_feats, dim=-1) # B T P C\n        corr = torch.einsum('bchw, btpc -> bpthw', img_feats, text_feats)\n        logit_scale = torch.clamp(logit_scale.exp(), max=100)\n        # corr = logit_scale * corr\n        return corr\n\n    def corr_embed(self, x):\n        B = x.shape[0]\n        corr_embed = rearrange(x, 'B P T H W -> (B T) P H W')\n        corr_embed = self.conv1(corr_embed)\n        corr_embed = rearrange(corr_embed, '(B T) C H W -> B C T H W', B=B)\n        return corr_embed\n    \n    def corr_projection(self, x, proj):\n        corr_embed = rearrange(x, 'B C T H W -> B T H W C')\n        corr_embed = proj(corr_embed)\n        corr_embed = rearrange(corr_embed, 'B T H W C -> B C T H W')\n        return corr_embed\n\n    def upsample(self, x):\n        B = x.shape[0]\n        corr_embed = rearrange(x, 'B C T H W -> (B T) C H W')\n        corr_embed = F.interpolate(corr_embed, scale_factor=2, mode='bilinear', align_corners=True)\n        corr_embed = rearrange(corr_embed, '(B T) C H W -> B C T H W', B=B)\n        return corr_embed\n\n    def conv_decoder(self, x, text_guidance, guidance, corr_guidance):\n        B = x.shape[0]\n        corr_embed = rearrange(x, 'B C T H W -> (B T) C H W')\n        mask_aux = self.head0(corr_embed.detach())\n        mask_aux = rearrange(mask_aux, '(B T) () H W -> B T H W', B=B)\n        corr_embed = self.decoder1(corr_embed, text_guidance[0], guidance[0], corr_guidance[0])\n        mask_aux0 = self.head1(corr_embed.detach())\n        mask_aux0 = rearrange(mask_aux0, '(B T) () H W -> B T H W', B=B)\n        corr_embed = self.decoder2(corr_embed, text_guidance[1], guidance[1], corr_guidance[1])\n        mask_aux1 = self.head2(corr_embed.detach())\n        mask_aux1 = rearrange(mask_aux1, '(B T) () H W -> B T H W', B=B)\n        corr_embed = self.decoder3(corr_embed, text_guidance[2], guidance[2], corr_guidance[2])\n        corr_embed = self.head(corr_embed)\n        corr_embed = rearrange(corr_embed, '(B T) () H W -> B T H W', B=B)\n        return corr_embed, mask_aux, mask_aux0, mask_aux1\n\n    def fast_conv_decoder(self, x, text_guidance, guidance, corr_guidance, topK):\n        B = x.shape[0]\n        T = x.shape[2]\n        corr_embed = rearrange(x, 'B C T H W -> (B T) C H W')\n        mask_aux = self.head0(corr_embed)\n        mask_aux = rearrange(mask_aux, '(B T) () H W -> B T H W', B=B)\n        aux_valid = self.get_valid_idx_from_mask(mask_aux, topK)\n        corr_embed = self.decoder1(corr_embed[aux_valid], text_guidance[0], guidance[0], corr_guidance[0][aux_valid])\n        mask_aux0 = self.head1(corr_embed)\n        mask_aux0 = rearrange(mask_aux0, '(B T) () H W -> B T H W', B=B)\n        aux0_valid = self.get_valid_idx_from_mask(mask_aux0, topK)\n        corr_embed = self.decoder2(corr_embed[aux0_valid], text_guidance[1], guidance[1], corr_guidance[1][aux_valid][aux0_valid])\n        mask_aux1 = self.head2(corr_embed)\n        mask_aux1 = rearrange(mask_aux1, '(B T) () H W -> B T H W', B=B)\n        aux1_valid = self.get_valid_idx_from_mask(mask_aux1, topK)\n        corr_embed = self.decoder3(corr_embed[aux1_valid], text_guidance[2], guidance[2], corr_guidance[2][aux_valid][aux0_valid][aux1_valid])\n        corr_embed = self.head(corr_embed)\n        corr_embed = rearrange(corr_embed, '(B T) () H W -> B T H W', B=B)\n        valid_idx = aux_valid.clone()\n        valid_idx1 = aux0_valid.clone()\n        valid_idx1[aux0_valid] = aux1_valid\n        valid_idx[aux_valid] = valid_idx1\n        _, _, H, W = corr_embed.size()\n        output = torch.zeros([B, T, H, W]).cuda()\n        output[:, valid_idx] = corr_embed.sigmoid()\n        return [output]\n\n    def get_valid_idx_from_mask(self, mask, topK=2):\n        T = mask.size(1)\n        mask = torch.argsort(mask, dim=1, descending=True)\n        mask = mask[:, :topK].reshape(-1)\n        valid_idx = torch.bincount(mask, minlength=T) > 0\n        return valid_idx\n\n    def forward(self, img_feats, text_feats, appearance_guidance, logit_scale):\n        \"\"\"\n        Arguments:\n            img_feats: (B, C, H, W)\n            text_feats: (B, T, P, C)\n            apperance_guidance: tuple of (B, C, H, W)\n        \"\"\"\n        # img_feats = self.clip_vis_projection(img_feats)\n        corr = self.correlation(img_feats, text_feats, logit_scale)\n        #corr = self.feature_map(img_feats, text_feats)\n        corr_embed = self.corr_embed(corr)\n        \n        projected_guidance, projected_text_guidance, projected_decoder_guidance = None, [None, None, None], [None, None]\n        projected_corr_decoder_guidance = [None, None, None]\n        # if self.guidance_projection is not None:\n        #     projected_guidance = self.guidance_projection(appearance_guidance[0])\n        if self.decoder_guidance_projection is not None:\n            projected_decoder_guidance = [proj(g) for proj, g in zip(self.decoder_guidance_projection, appearance_guidance[1:])]\n\n        if self.decoder_corr_guidance_projection is not None:\n            corr = rearrange(corr, 'B P T H W -> (B T) P H W')\n            for i, proj in enumerate(self.decoder_corr_guidance_projection):\n                corr = proj(corr)\n                projected_corr_decoder_guidance[i] = corr\n\n        if self.text_guidance_projection is not None:\n            text_feats = text_feats.mean(dim=-2)\n            text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)\n            projected_text_guidance = [proj(text_feats) for proj in self.text_guidance_projection]\n        if self.fast_infer and not self.training:\n            logit = self.fast_conv_decoder(corr_embed, projected_text_guidance, projected_decoder_guidance, projected_corr_decoder_guidance, topK=self.topK)\n        else:\n            logit = self.conv_decoder(corr_embed, projected_text_guidance, projected_decoder_guidance, projected_corr_decoder_guidance)\n\n        return logit\n")

print('✅ Cross-Modal Guided Filter (CM-GF) and configs cleanly injected into SED repository!')

In [ ]:
# Cell 5: Run Official Evaluation for BASELINE SED (CVPR 2024)
print('=' * 80)
print('🚀 RUNNING OFFICIAL EVALUATION: BASELINE SED (ConvNeXt-B on ADE20K-150)')
print('=' * 80)
!python train_net.py --config-file configs/convnextB_768.yaml --eval-only \
    MODEL.WEIGHTS sed_convnext_b.pth \
    MODEL.SEM_SEG_HEAD.TEST_CLASS_JSON datasets/ade150.json \
    DATASETS.TEST '("ade20k_150_test_sem_seg",)' \
    MODEL.SEM_SEG_HEAD.POOLING_SIZES '[1,1]' \
    MODEL.SEM_SEG_HEAD.USE_CM_GF False
print('✅ Baseline SED evaluation complete.')

In [ ]:
# Cell 6: Run Official Evaluation for PROPOSED SED + CM-GF
print('=' * 80)
print('🚀 RUNNING OFFICIAL EVALUATION: PROPOSED SED + CM-GF (ConvNeXt-B on ADE20K-150)')
print('=' * 80)
!python train_net.py --config-file configs/convnextB_768.yaml --eval-only \
    MODEL.WEIGHTS sed_convnext_b.pth \
    MODEL.SEM_SEG_HEAD.TEST_CLASS_JSON datasets/ade150.json \
    DATASETS.TEST '("ade20k_150_test_sem_seg",)' \
    MODEL.SEM_SEG_HEAD.POOLING_SIZES '[1,1]' \
    MODEL.SEM_SEG_HEAD.USE_CM_GF True
print('✅ Proposed SED + CM-GF evaluation complete.')

### 📸 What to Screenshot:
Take a screenshot of the final printed evaluation summary lines from Cell 5 and Cell 6:
- `copypaste: Task: sem_seg`
- `copypaste: mIoU: 31.8%` vs. `mIoU: 32.X%`
Paste them into the chat for final presentation integration!